In [ ]:
from sqlalchemy import (
    Boolean,
    Column,
    Float,
    Integer,
    MetaData,
    String,
    Table,
    text,
    update,
)

from egomimic.utils.aws.aws_sql import (
    TableRow,
    create_default_engine,
    episode_table_to_df,
)

In [ ]:
engine = create_default_engine()

In [ ]:
df = episode_table_to_df(engine)
df

## Example Useful Functions

In [ ]:
# Add Episode Test
episode = TableRow(
    episode_hash=1761408819,
    operator="test",
    lab="lab-x",
    num_frames=-1,
    task="bimanual_test",
    task_description="Dummy row for table inspection",
    scene="kitchen",
    objects="cup,plate,spoon",
    processed_path="",
    zarr_processed_path="",
    zarr_processing_error="",
    zarr_mp4_path="",
    mp4_path="",
    embodiment="aria",
    robot_name="",
    is_eval=False,
    eval_score=0.0,
    eval_success=False,
)
# Adding an episode
# add_episode(engine, episode)

# Update Episode Test
# episode.operator = "simar"
# update_episode(engine, episode)

# Get Table Row from Episode Hash
# episode_hash_to_table_row(engine, 123456)

# Delete episodes by hash
# delete_episodes(engine, [123456, 123457])

In [ ]:
def drop_table(table_name):
    with engine.connect() as connection:
        connection.execute(text(f"DROP TABLE IF EXISTS app.{table_name} CASCADE;"))
        connection.commit()
    print(f"Dropped table '{table_name}' from schema 'app' if it existed.")

In [ ]:
def create_table():
    metadata = MetaData(schema="app")

    Table(
        "episodes",
        metadata,
        Column("episode_hash", String, primary_key=True),
        Column("operator", String),
        Column("lab", String),
        Column("num_frames", Integer),
        Column("task", String),
        Column("task_description", String),
        Column("scene", String),
        Column(
            "objects", String
        ),  # Store as JSON or comma-separated list of object names
        Column("processed_path", String),
        Column("zarr_processed_path", String),
        Column("zarr_processing_error", String),
        Column("zarr_mp4_path", String),
        Column("mp4_path", String),
        Column("is_deleted", Boolean),
        Column("embodiment", String),
        Column("robot_name", String),
        Column("is_eval", Boolean),
        Column("eval_score", Float),
        Column("eval_success", Boolean),
    )

    metadata.create_all(engine)
    print("Created table 'episodes' in schema 'app'.")

In [ ]:
def delete_episodes_by_task(task_name: str):
    episodes_tbl = Table("episodes", MetaData(), autoload_with=engine, schema="app")
    stmt = (
        update(episodes_tbl)
        .where(episodes_tbl.c.task == task_name)
        .values(is_deleted=True)
    )
    with engine.begin() as conn:
        conn.execute(stmt)